# Atlas Trade — Unsloth Qwen3.5-27B Setup

**Runtime:** Google Colab Pro — A100 80GB  
**Model:** `unsloth/Qwen3.5-27B` (BF16, no quantization)  
**Context:** 16384 tokens

Run cells top to bottom on a fresh A100 runtime. On subsequent sessions, re-run from **Step 4** onwards (model is cached in HF cache after the first download).

## Configuration

Set your GitHub repo URL and Hugging Face token here, or export them as environment variables before running:

```python
import os
os.environ["PROJECT_REPO_URL"] = "https://github.com/<your-username>/unsloth-qwen35-trading-assistant.git"
os.environ["HF_TOKEN"] = "hf_..."
```

In [2]:
import os

REPO_URL = os.environ.get(
    "PROJECT_REPO_URL",
    "https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant.git",
)
HF_TOKEN = os.environ.get("HF_TOKEN", "")

MODEL_NAME      = "unsloth/Qwen3.5-27B"
MAX_SEQ_LENGTH  = 8192   # 16384 -> 8192: KV cache yari yariya kuculus
DTYPE           = None   # None = auto-detect (BF16 on A100)
LOAD_IN_4BIT    = True   # 4-bit quant: VRAM ~54GB -> ~14GB

REPO_DIR = "/content/unsloth-qwen35-trading-assistant"

print(f"Model   : {MODEL_NAME}")
print(f"Context : {MAX_SEQ_LENGTH} tokens")
print(f"4-bit   : {LOAD_IN_4BIT}")
print(f"Repo    : {REPO_URL}")
print(f"HF token: {chr(39)}set{chr(39) if HF_TOKEN else chr(39) + chr(39)}")

Model   : unsloth/Qwen3.5-27B
Context : 8192 tokens
4-bit   : True
Repo    : https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant.git
HF token: 'set''


## Step 1 — Clone or update the repository

In [3]:
import subprocess, pathlib, sys

def _run(cmd: list[str], **kwargs) -> str:
    result = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    output = (result.stdout + result.stderr).strip()
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}\n{output}")
    return output

def git_pull_latest():
    """Clone the repo on first run; fast-forward pull on subsequent runs."""
    repo = pathlib.Path(REPO_DIR)
    if repo.exists():
        print(_run(["git", "-C", str(repo), "pull", "--ff-only"]))
    else:
        print(_run(["git", "clone", REPO_URL, str(repo)]))
    # Add repo to Python path so `src` is importable
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    print(f"Repo ready at {repo}")

git_pull_latest()

Updating 50232d4..106eb22
Fast-forward
 setup.ipynb | 1016 ++++++++++++++++++++++++-----------------------------------
 1 file changed, 412 insertions(+), 604 deletions(-)
From https://github.com/umiterkanetc-art/unsloth-qwen35-trading-assistant
   50232d4..106eb22  main       -> origin/main
Repo ready at /content/unsloth-qwen35-trading-assistant


## Step 2 — Install Unsloth

Uses the official runtime-aware auto-installer — automatically matches the active CUDA/Torch stack on this Colab runtime.

In [4]:
import subprocess, sys

def _pip(*args):
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args],
        capture_output=True, text=True
    )
    # Print last 2000 chars of output so failures are visible
    out = (result.stdout + result.stderr)[-2000:]
    print(out)
    return result.returncode

print("=== Installing Unsloth ===\n")
code = _pip("install", "unsloth")

if code != 0:
    print("\nStandard install failed, trying git source...")
    code = _pip(
        "install",
        "unsloth @ git+https://github.com/unslothai/unsloth.git",
        "--no-build-isolation",
    )

if code != 0:
    raise RuntimeError("Unsloth installation failed. See output above.")

print("\n✅ Unsloth installed successfully.")
print("\n⚠️  RESTART THE RUNTIME NOW, then re-run from Step 1 (skip this cell).")
print("   Runtime menu → Restart session")

=== Installing Unsloth ===

l/lib/python3.12/dist-packages (from pandas->datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1->unsloth) (2025.2)


✅ Unsloth installed successfully.

⚠️  RESTART THE RUNTIME NOW, then re-run from Step 1 (skip this cell).
   Runtime menu → Restart session


## Step 3 — Hugging Face login

Required only if you plan to push LoRA adapters or access gated models. Skip if `HF_TOKEN` is empty.

In [5]:
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub.")
else:
    print("No HF_TOKEN set — skipping login (public model download will still work).")

No HF_TOKEN set — skipping login (public model download will still work).


## Step 4 — Load the model

First run downloads ~54 GB of BF16 weights. Subsequent runs load from the HF cache (~1-2 minutes).

In [6]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

FastLanguageModel.for_inference(model)

print(f"Model loaded: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.4: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/1184 [00:00<?, ?it/s]

Model loaded: unsloth/Qwen3.5-27B
Max sequence length: 8192


## Step 5 — Chat function

`chat()` keeps a conversation history so Atlas Trade remembers the context within a session.  
Call `chat(reset=True)` to start a fresh conversation.

In [7]:
import sys, re
sys.path.insert(0, REPO_DIR)

from transformers import TextStreamer
from src.trading_prompt import TRADING_SYSTEM_PROMPT, build_trading_messages

_conversation_history = []
_text_tokenizer = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
_streamer = TextStreamer(_text_tokenizer, skip_prompt=True, skip_special_tokens=True)

def _strip_thinking(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

def chat(
    user_message,
    market_context="",
    reset=False,
    thinking=False,
    max_new_tokens=1024,
    temperature=0.6,
    top_p=0.95,
    stream=True,
):
    global _conversation_history
    if reset:
        _conversation_history = []
        print("[sifirlandı]")

    # /no_think: Qwen3 thinking modunu kapatir (hizli, Turkce, dogrudan cevap)
    # /think  : derin analiz gerektiginde acilir
    message = user_message + (" /think" if thinking else " /no_think")

    messages = build_trading_messages(
        user_message=message,
        market_context=market_context,
        history=_conversation_history,
    )

    text = _text_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = _text_tokenizer(text, return_tensors="pt").to("cuda")
    input_ids      = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        pad_token_id=_text_tokenizer.eos_token_id,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        do_sample=True,
        streamer=_streamer if stream else None,
    )

    new_tokens = outputs[0][input_ids.shape[1]:]
    response = _text_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    response = _strip_thinking(response)

    _conversation_history.append({"role": "user",      "content": message})
    _conversation_history.append({"role": "assistant", "content": response})
    return response

def clear_history():
    global _conversation_history
    _conversation_history = []
    print("[gecmis temizlendi]")

print("chat() hazir")


chat() hazir


## Step 6 — Start trading

Atlas Trade hazır. Aşağıdaki hücreyi düzenleyerek analizini başlat.

## Step 7 — Live market data (Bybit)

`fetch_context()` Bybit'ten canlı OHLCV çeker, RSI / EMA / MACD hesaplar ve `chat()` için hazır bir `market_context` stringi döner.  
API key gerekmez — public endpoint.

In [8]:
import importlib
import src.market_data as _md
importlib.reload(_md)
from src.market_data import fetch_context

def live_chat(user_message, symbol="BTCUSDT", timeframe="4h", reset=False, **kwargs):
    """Bybit canli veri cekip chat()'e otomatik gonder."""
    print(f"[{symbol} {timeframe.upper()} verisi aliniyor...]")
    ctx = fetch_context(symbol, timeframe=timeframe)
    print(ctx)
    print("-" * 50)
    return chat(user_message, market_context=ctx, reset=reset, **kwargs)

print("live_chat() hazir")
print("  live_chat('Bu setup long icin uygun mu?')")
print("  live_chat('ETH analiz et', symbol='ETHUSDT', timeframe='1h')")


live_chat() hazir
  live_chat('Bu setup long icin uygun mu?')
  live_chat('ETH analiz et', symbol='ETHUSDT', timeframe='1h')


In [9]:
# --- Canlı veriyle analiz ---
response = chat(
    user_message="Bu setup nasıl, long için uygun mu?",
    market_context=fetch_context("BTCUSDT", timeframe="4h"),
    reset=True,
)
print(response)

[sifirlandı]
Thinking Process:

1.  **Analyze the Request:**
    *   **User:** Asks "Bu setup nasıl, long için uygun mu?" (How is this setup, is it suitable for long?)
    *   **Context:** BTCUSDT, 4H Timeframe.
    *   **Data Provided:**
        *   Price: 72,700.00 USDT (+0.37% last candle).
        *   EMAs: Price > EMA20 > EMA50 > EMA200 (Bullish alignment).
        *   RSI: 69.5 (Approaching overbought but not there yet).
        *   MACD: Positive, rising, bullish momentum.
        *   Volume: Low (0.2x of 20-candle average).
        *   Futures: Funding positive (0.0093%), Open Interest high.
    *   **Constraint:** `/no_think` (This is a system instruction override, but as an AI model, I must still follow my core instructions regarding analysis quality and safety. However, the user explicitly requested `/no_think` which usually implies skipping the internal reasoning block in the output. I need to balance the persona of "Atlas Trade" with the instruction. Since I am a model, I 

In [10]:
response = chat(
    user_message="Merhaba, BTC/USDT 4H chart'ta trend durumu nasıl görünüyor?",
    market_context="""
Sembol  : BTC/USDT
Timeframe: 4H
Fiyat   : 83,500 USDT
RSI(14) : 54
MACD    : histogram pozitif, sinyal çizgisinin üzerinde
EMA20   : 82,100 (fiyat üzerinde)
EMA50   : 79,800 (fiyat üzerinde)
Hacim   : ortalama üzerinde son 3 mumda
""",
)

print(response)

Thinking Process:

1.  **Analyze the Request:**
    *   **User:** Asks about the trend status on BTC/USDT 4H chart.
    *   **Input Data:**
        *   Symbol: BTC/USDT
        *   Timeframe: 4H
        *   Price: 83,500 USDT
        *   RSI(14): 54 (Neutral/Bullish momentum)
        *   MACD: Histogram positive, above signal line (Bullish momentum)
        *   EMA20: 82,100 (Price > EMA20)
        *   EMA50: 79,800 (Price > EMA50)
        *   Volume: Above average on last 3 candles.
    *   **Constraint:** `/no_think` (Do not output internal reasoning chain, just the final response).
    *   **Role:** Atlas Trade (Institutional-style crypto trading analysis assistant).
    *   **Language:** Turkish.
    *   **Goal:** Provide a structured market read and trend analysis based on the provided data.

2.  **Evaluate the Data:**
    *   **Trend:** Price is above EMA20 and EMA50. EMA20 > EMA50. This is a clear bullish alignment (Golden Cross structure or at least bullish stacking).
    *   *

In [ ]:
# Devam sorusu — geçmiş otomatik taşınır
response = chat("Bu durumda long için en temiz entry bölgesi neresi?")
print(response)

Thinking Process:

1.  **Analyze the Request:**
    *   **User:** Asks "Bu durumda long için en temiz entry bölgesi neresi?" (Where is the cleanest entry zone for long in this situation?)
    *   **Context:** Continuing from the previous turn (BTC/USDT 4H chart).
    *   **Data Available (from previous turn):**
        *   Price: 83,500 USDT
        *   EMA20: 82,100
        *   EMA50: 79,800
        *   RSI: 54
        *   MACD: Positive, above signal
        *   Volume: Above average (last 3 candles)
        *   Trend: Bullish (Price > EMA20 > EMA50)
    *   **Constraint:** `/no_think` (Do not output internal reasoning chain).
    *   **Role:** Atlas Trade (Institutional-style crypto trading analysis assistant).
    *   **Language:** Turkish.
    *   **Goal:** Provide a specific, actionable entry 

zone based on 

KeyboardInterrupt: 

---

## Utility — Update repo without reloading the model

In [ ]:
# Run this cell whenever you push prompt or code changes from VS Code.
# The model stays loaded — only the Python source is refreshed.
git_pull_latest()

# Reload the trading_prompt module to pick up any changes
import importlib, src.trading_prompt as _tp
importlib.reload(_tp)
from src.trading_prompt import TRADING_SYSTEM_PROMPT, build_trading_messages
print("Prompt module reloaded.")

Already up to date.
Repo ready at /content/unsloth-qwen35-trading-assistant
Prompt module reloaded.
